# Generación de .json para mayor contexto

A fin de generar mejores imágenes basándonos en las respuestas del RAG, hemos escrito este código. En él, se hará lo siguiente:

- En el apartado de NLP hemos obtenido lo siguiente:

    - Un .json con 100 preguntas respondidas por el RAG. En el .json, tenemos columnas como la respuesta del RAG, la respuesta esperada, el capítulo del libro en el que se encuentra la respuesta, los chunks que ha elegido el retrieval... entre otras columnas.
    - A nosotros las que nos interesan ahora mismo son dos columnas: la del capítulo esperado y la de los chunks seleccionados por el retrieval.

- A la hora de diseñar el RAG, se crearon dos colecciones:
    - La primera colección utiliza el libro como fuente. Los chunks se hicieron respetando la estructura del libro y cogiendo chunks de 6000 caracteres.
    - La segunda colección se creó haciendo Web Scrapping de una página web de fandom. Se scrappearon resúmenes de cada capítulo del libro. Por como estaban estructurados los resúmenes, se decidió hacer un chunk por parrafo del resumen, pues cada parrafo del resumen de un mismo capítulo cuenta algo difente dentro de un mismo capítulo.

- Teniendo estas dos cosas en cuenta, hemos planteado lo siguiente:

    - A fin de tener un mayor contexto en la generación de imágenes, hemos decidido que vamos a utilizar el chunk de la colección de resúmenes que mejor se adapte a la pregunta a responder y por tanto a la pregunta a dibujar.

    - Para ello, utilizando la columna del capítulo en el que está la respuesta y la columna de los chunks elegido por el retrieval, vamos a hacer lo siguiente:

        - Primero, vemos cual es el capitulo en el que está la respuesta, y vemos si, entre los chunks del retrieval, se encuentra algún chunk perteneciente a ese capítulo. Por la evaluación llevada a cabo en el trabajo de NLP, sabemos que un alto porcentaje tienen el chunk correcto para responder.

        - Una vez seleccionado el chunk perteneciente al capítulo esperado, y por tanto, el chunk del que se puede extraer la información para responder a la pregunta, la idea es buscar el chunk más parecido en la colección de los resúmenes. Es decir, si en un capitulo se cuenta tres "tramas" (trama A, trama B y trama C), y la trama necesaria para responder a la pregunta es la B, vamos a buscar en la colección de resúmenes el parrafo correspondiente a la trama B.
    - Para llevar acabo esta tarea, utilizaremos embeddings. Embeddearemos el chunk necesario y buscaremos por distancia coseno el chunk en la colección de resúmenes más cercano. Este chunk será luego utilizado para tener más contexto a la hora de crear imágenes.

Con todo esto claro, el resultado final será un .json que posteriormente podremos utilizar para tener más contexto. El código necesario para esto es el que se encuentra abajo.

In [2]:
!pip install -q -U chromadb sentence-transformers pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 115.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 155.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 129.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [3]:
import json
import re
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import chromadb

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
JSONL_PATH = Path("/content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs/q&a_fixed/gemma4_31b_4bit_fixed.jsonl")

CHROMA_RESUMENES_PATH = Path("/content/drive/MyDrive/NLP_PRACTICA/Persist/chroma_db_resumenes_qwen")
COLLECTION_RESUMENES = "resumenes_got"

EMBEDDING_MODEL_ID = "Qwen/Qwen3-Embedding-8B"

OUTPUT_JSONL = Path("/content/drive/MyDrive/Proyecto NLP y DL/Imagen_Generacion/chunks_para_resumen.jsonl")
OUTPUT_JSON =  Path("/content/drive/MyDrive/Proyecto NLP y DL/Imagen_Generacion/chunks_para_resumen.json")


In [5]:
def normalize_text(x):
    if x is None:
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows



In [6]:
def gold_chapter_to_order(gold_chapter):
    """
    Converts:
    - 'Prólogo' -> 0
    - 'Capítulo 1' -> 1
    - 'Capitulo 1' -> 1
    """
    gold = normalize_text(gold_chapter).lower()

    if "prólogo" in gold or "prologo" in gold:
        return 0

    m = re.search(r"cap[íi]tulo\s+(\d+)", gold)
    if m:
        return int(m.group(1))

    return None


def get_gold_chapters(row):
    gold = row.get("gold_chapters", [])
    if isinstance(gold, str):
        gold = [gold]
    return gold


def get_gold_chapter_orders(row):
    orders = []

    for g in get_gold_chapters(row):
        order = gold_chapter_to_order(g)
        if order is not None:
            orders.append(order)

    return orders


def summary_chapter_to_order(summary_chapter):
    """
    Converts:
    - 'Juego de Tronos-Prólogo' -> 0
    - 'Juego de Tronos-Capítulo 1' -> 1
    """
    ch = normalize_text(summary_chapter).lower()

    if "prólogo" in ch or "prologo" in ch:
        return 0

    m = re.search(r"cap[íi]tulo\s+(\d+)", ch)
    if m:
        return int(m.group(1))

    return None


In [7]:
def find_matching_retrieved_chunk(row):
    """
    Finds the retrieved chunk whose metadata.chapter_order matches the golden chapter.
    """
    gold_orders = get_gold_chapter_orders(row)
    retrieved_chunks = row.get("retrieved_chunks", []) or []

    for chunk in retrieved_chunks:
        meta = chunk.get("metadata", {}) or {}
        chunk_order = meta.get("chapter_order")

        try:
            chunk_order = int(chunk_order)
        except Exception:
            chunk_order = None

        if chunk_order in gold_orders:
            return chunk

    return None


In [8]:
def get_all_summary_chunks(collection):
    result = collection.get(include=["documents", "metadatas"])

    chunks = []
    for doc_id, doc, meta in zip(
        result.get("ids", []),
        result.get("documents", []),
        result.get("metadatas", [])
    ):
        meta = meta or {}

        chunks.append({
            "id": doc_id,
            "text": doc,
            "metadata": meta,
            "chapter_order": summary_chapter_to_order(meta.get("chapter"))
        })

    return chunks


def get_summary_candidates_for_gold_chapter(all_summary_chunks, gold_orders):
    return [
        c for c in all_summary_chunks
        if c.get("chapter_order") in gold_orders
    ]



In [9]:
def find_best_summary_chunk(model, book_chunk_text, summary_candidates):
    if not summary_candidates:
        return None

    book_emb = model.encode(
        book_chunk_text,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    summary_texts = [c["text"] for c in summary_candidates]

    summary_embs = model.encode(
        summary_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=4,
        show_progress_bar=False
    )

    sims = summary_embs @ book_emb
    best_idx = int(np.argmax(sims))

    best = summary_candidates[best_idx].copy()
    best["similarity_score"] = float(sims[best_idx])

    return best


In [13]:
rows = read_jsonl(JSONL_PATH)
print(f"Filas cargadas: {len(rows)}")

client = chromadb.PersistentClient(path=str(CHROMA_RESUMENES_PATH))
col_resumenes = client.get_collection(COLLECTION_RESUMENES)

all_summary_chunks = get_all_summary_chunks(col_resumenes)
print(f"Chunks de resumen cargados: {len(all_summary_chunks)}")

model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    trust_remote_code=True,
    device="cuda"
)


Filas cargadas: 105
Chunks de resumen cargados: 320


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [14]:

ROMAN_MAP = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
    "VII": 7,
    "VIII": 8,
    "IX": 9,
    "X": 10,
    "XI": 11,
    "XII": 12,
    "XIII": 13,
    "XIV": 14,
    "XV": 15,
}

POV_TO_CHAPTER_ORDERS = {}

for chunk in all_summary_chunks:
    # Esto no sirve para POV, así que lo haremos desde retrieved_chunks del JSONL
    pass

for row in rows:
    for chunk in row.get("retrieved_chunks", []) or []:
        meta = chunk.get("metadata", {}) or {}
        pov = meta.get("pov")
        chapter_title = meta.get("chapter_title")
        chapter_order = meta.get("chapter_order")

        if pov is None or chapter_title is None or chapter_order is None:
            continue

        m = re.search(r"\((\d+)\)", str(chapter_title))
        if not m:
            continue

        pov_key = str(pov).strip().upper()
        pov_num = int(m.group(1))

        POV_TO_CHAPTER_ORDERS[(pov_key, pov_num)] = int(chapter_order)


def roman_to_int(x):
    x = normalize_text(x).upper()
    return ROMAN_MAP.get(x)


def gold_chapter_to_order(gold_chapter):
    """
    Converts:
    - 'Prólogo' -> 0
    - 'Bran I' -> 1
    - 'Catelyn II' -> ...
    """
    gold = normalize_text(gold_chapter)

    if "prólogo" in gold.lower() or "prologo" in gold.lower():
        return 0

    # If already comes as "Capítulo 1"
    m = re.search(r"cap[íi]tulo\s+(\d+)", gold.lower())
    if m:
        return int(m.group(1))

    # Format: "Bran I", "Eddard II", "Daenerys III"
    m = re.search(r"^([A-Za-zÁÉÍÓÚÑÜáéíóúñü]+)\s+([IVXLCDM]+)$", gold.strip(), re.IGNORECASE)
    if m:
        pov = m.group(1).upper()
        roman = m.group(2).upper()
        pov_num = roman_to_int(roman)

        return POV_TO_CHAPTER_ORDERS.get((pov, pov_num))

    return None

In [15]:
outputs = []

for row in tqdm(rows):
    idx = row.get("idx")
    question = row.get("question")
    gold_chapters = get_gold_chapters(row)
    gold_orders = get_gold_chapter_orders(row)

    # Skip prologue rows
    if 0 in gold_orders:
        outputs.append({
            "idx": idx,
            "question": question,
            "gold_chapters": gold_chapters,
            "gold_chapter_orders": gold_orders,
            "status": "skipped_prologue",
            "matched_book_chunk": None,
            "num_summary_candidates_same_chapter": 0,
            "best_summary_chunk": None,
        })
        continue

    matched_book_chunk = find_matching_retrieved_chunk(row)

    if matched_book_chunk is None:
        outputs.append({
            "idx": idx,
            "question": question,
            "gold_chapters": gold_chapters,
            "gold_chapter_orders": gold_orders,
            "status": "no_matching_retrieved_chunk",
            "matched_book_chunk": None,
            "num_summary_candidates_same_chapter": 0,
            "best_summary_chunk": None,
        })
        continue

    book_chunk_text = matched_book_chunk.get("text", "")

    summary_candidates = get_summary_candidates_for_gold_chapter(
        all_summary_chunks,
        gold_orders
    )

    if not summary_candidates:
        outputs.append({
            "idx": idx,
            "question": question,
            "gold_chapters": gold_chapters,
            "gold_chapter_orders": gold_orders,
            "status": "no_summary_candidates_for_gold_chapter",
            "matched_book_chunk": matched_book_chunk,
            "num_summary_candidates_same_chapter": 0,
            "best_summary_chunk": None,
        })
        continue

    best_summary_chunk = find_best_summary_chunk(
        model=model,
        book_chunk_text=book_chunk_text,
        summary_candidates=summary_candidates
    )

    outputs.append({
        "idx": idx,
        "question": question,
        "gold_chapters": gold_chapters,
        "gold_chapter_orders": gold_orders,
        "status": "ok",
        "matched_book_chunk": matched_book_chunk,
        "num_summary_candidates_same_chapter": len(summary_candidates),
        "best_summary_chunk": best_summary_chunk,
    })



  0%|          | 0/105 [00:00<?, ?it/s]

In [16]:
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for item in outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)

print("Guardado JSONL en:", OUTPUT_JSONL)
print("Guardado JSON en:", OUTPUT_JSON)

Guardado JSONL en: /content/drive/MyDrive/Proyecto NLP y DL/Imagen_Generacion/chunks_para_resumen.jsonl
Guardado JSON en: /content/drive/MyDrive/Proyecto NLP y DL/Imagen_Generacion/chunks_para_resumen.json


In [17]:
ok = sum(1 for x in outputs if x["status"] == "ok")
skipped_prologue = sum(1 for x in outputs if x["status"] == "skipped_prologue")
no_book = sum(1 for x in outputs if x["status"] == "no_matching_retrieved_chunk")
no_summary = sum(1 for x in outputs if x["status"] == "no_summary_candidates_for_gold_chapter")

print()
print("Resumen:")
print("OK:", ok)
print("Saltados por prólogo:", skipped_prologue)
print("Sin chunk del libro coincidente:", no_book)
print("Sin chunks de resumen del capítulo golden:", no_summary)


Resumen:
OK: 89
Saltados por prólogo: 4
Sin chunk del libro coincidente: 12
Sin chunks de resumen del capítulo golden: 0
